# 04 - Feature Engineering

Create the final content feature matrix for Spotify track similarity.

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import save_npz
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"

TRACKS_PATH = PROCESSED_DIR / "content_tracks_preprocessed.csv"
CONFIG_PATH = MODEL_DIR / "content_feature_config.json"

## Load Preprocessed Tracks

In [2]:
tracks = pd.read_csv(TRACKS_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

numeric_features = feature_config["numeric_features"]
categorical_features = feature_config["categorical_features"]

print(f"Tracks: {tracks.shape[0]:,} rows x {tracks.shape[1]:,} columns")

Tracks: 89,740 rows x 21 columns


## Engineer Track Features

In [3]:
features = tracks.copy()

features["duration_minutes"] = features["duration_ms"] / 1000 / 60
features["tempo_log"] = np.log1p(features["tempo"])
features["energy_danceability"] = features["energy"] * features["danceability"]
features["mood_score"] = (features["valence"] + features["energy"]) / 2
features["acoustic_energy_balance"] = features["acousticness"] - features["energy"]
features["instrumental_acoustic_score"] = features["instrumentalness"] * features["acousticness"]
features["popularity_percentile"] = features["popularity"].rank(pct=True)

features["popularity_level"] = pd.cut(
    features["popularity"], [-1, 20, 45, 70, 100], labels=["low", "medium", "high", "very_high"]
)
features["tempo_group"] = pd.cut(
    features["tempo"], [0, 90, 120, 150, np.inf], labels=["slow", "medium", "fast", "very_fast"]
)
features["energy_level"] = pd.cut(
    features["energy"], [-0.01, 0.33, 0.66, 1.0], labels=["low_energy", "medium_energy", "high_energy"]
)
features["danceability_level"] = pd.cut(
    features["danceability"], [-0.01, 0.33, 0.66, 1.0], labels=["low_dance", "medium_dance", "high_dance"]
)
features["acoustic_profile"] = np.where(features["acousticness"] >= 0.5, "acoustic", "produced")
features["vocal_profile"] = np.where(features["instrumentalness"] >= 0.5, "instrumental", "vocal")

features["mood_label"] = np.select(
    [
        (features["valence"] >= 0.55) & (features["energy"] >= 0.55),
        (features["valence"] >= 0.55) & (features["energy"] < 0.55),
        (features["valence"] < 0.55) & (features["energy"] >= 0.55),
    ],
    ["bright_energetic", "calm_positive", "intense_dark"],
    default="calm_dark",
)

engineered_numeric = [
    "duration_minutes", "tempo_log", "energy_danceability", "mood_score",
    "acoustic_energy_balance", "instrumental_acoustic_score", "popularity_percentile",
]
engineered_categorical = [
    "popularity_level", "tempo_group", "energy_level", "danceability_level",
    "acoustic_profile", "vocal_profile", "mood_label",
]

features[engineered_numeric + engineered_categorical].head()

,duration_minutes,tempo_log,energy_danceability,mood_score,acoustic_energy_balance,instrumental_acoustic_score,popularity_percentile,popularity_level,tempo_group,energy_level,danceability_level,acoustic_profile,vocal_profile,mood_label
0,3.844433,4.487703,0.311636,0.5880,-0.4288,3.252200e-08,0.978466,very_high,slow,medium_energy,high_dance,produced,vocal,calm_positive
1,2.493500,4.362958,0.069720,0.2165,0.7580,5.137440e-06,0.835408,high,slow,low_energy,medium_dance,acoustic,vocal,calm_dark
2,3.513767,4.348108,0.157242,0.2395,-0.1490,0.000000e+00,0.861143,high,slow,medium_energy,medium_dance,produced,vocal,calm_dark
3,3.365550,5.208064,0.015854,0.1013,0.8454,6.398350e-05,0.971573,very_high,very_fast,low_energy,low_dance,acoustic,vocal,calm_dark
4,3.314217,4.795369,0.273774,0.3050,0.0260,0.000000e+00,0.995938,very_high,medium,medium_energy,medium_dance,produced,vocal,calm_dark


## Add Artist and Genre Stats

In [4]:
artist_stats = features.groupby("artists").agg(
    artist_track_count=("track_id", "count"),
    artist_avg_popularity=("popularity", "mean"),
).reset_index()

genre_stats = features.groupby("track_genre").agg(
    genre_track_count=("track_id", "count"),
    genre_avg_popularity=("popularity", "mean"),
).reset_index()

features = features.merge(artist_stats, on="artists", how="left")
features = features.merge(genre_stats, on="track_genre", how="left")

aggregate_features = ["artist_track_count", "artist_avg_popularity", "genre_track_count", "genre_avg_popularity"]
features[aggregate_features].head()

,artist_track_count,artist_avg_popularity,genre_track_count,genre_avg_popularity
0,7,51.571429,1000,42.483
1,9,41.222222,1000,42.483
2,1,57.000000,1000,42.483
3,15,53.933333,1000,42.483
4,11,41.727273,1000,42.483


## Build Text Soup

In [5]:
text_columns = [
    "track_name", "artists", "album_name", "track_genre", "mood_label",
    "tempo_group", "energy_level", "danceability_level", "acoustic_profile", "vocal_profile",
]

text_frame = features[text_columns].astype("object").fillna("unknown").astype(str)
features["content_text"] = (
    text_frame.agg(" ".join, axis=1)
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

features[["track_name", "artists", "content_text"]].head()

,track_name,artists,content_text
0,Comedy,Gen Hoshino,comedy gen hoshino comedy acoustic calm positi...
1,Ghost - Acoustic,Ben Woodward,ghost acoustic ben woodward ghost acoustic aco...
2,To Begin Again,Ingrid Michaelson;ZAYN,to begin again ingrid michaelson zayn to begin...
3,Can't Help Falling In Love,Kina Grannis,can t help falling in love kina grannis crazy ...
4,Hold On,Chord Overstreet,hold on chord overstreet hold on acoustic calm...


## Create Feature Matrix

In [6]:
final_numeric_features = numeric_features + engineered_numeric + aggregate_features
final_categorical_features = categorical_features + engineered_categorical

feature_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), final_numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), final_categorical_features),
        ("text", TfidfVectorizer(max_features=4000, ngram_range=(1, 2), min_df=2), "content_text"),
    ],
    sparse_threshold=1.0,
)

feature_matrix = feature_preprocessor.fit_transform(features)
feature_names = feature_preprocessor.get_feature_names_out().tolist()

print(f"Feature matrix: {feature_matrix.shape}")
print(f"Features: {len(feature_names):,}")

Feature matrix: (89740, 4163)
Features: 4,163


## Save Feature Engineering Outputs

In [7]:
features_path = PROCESSED_DIR / "content_tracks_engineered.csv"
matrix_path = MODEL_DIR / "engineered_feature_matrix.npz"
preprocessor_path = MODEL_DIR / "engineered_feature_preprocessor.joblib"
feature_names_path = MODEL_DIR / "engineered_feature_names.json"

features.to_csv(features_path, index=False)
save_npz(matrix_path, feature_matrix)
joblib.dump(feature_preprocessor, preprocessor_path)

with open(feature_names_path, "w", encoding="utf-8") as file:
    json.dump(feature_names, file, indent=2)

print("Saved feature engineering outputs.")
print(features_path)
print(matrix_path)
print(preprocessor_path)

Saved feature engineering outputs.
d:\ml\sportify-recommendation\data\processed\content_tracks_engineered.csv
d:\ml\sportify-recommendation\models\content_based\engineered_feature_matrix.npz
d:\ml\sportify-recommendation\models\content_based\engineered_feature_preprocessor.joblib


## Summary

In [8]:
pd.DataFrame({
    "item": ["tracks", "engineered_columns", "matrix_rows", "matrix_features"],
    "value": [len(features), features.shape[1], feature_matrix.shape[0], feature_matrix.shape[1]],
})

,item,value
0,tracks,89740
1,engineered_columns,40
2,matrix_rows,89740
3,matrix_features,4163
